In [1]:
from pathlib import Path
import os
import random
import shutil
import torch
import kagglehub

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

c:\Users\cooki\Documents\GitHub\FOCA\Source\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.0+cu130
CUDA disponível: True
GPU: NVIDIA GeForce MX570 A
VRAM: 2.0 GB


In [3]:
BASE_DIR = Path.cwd()

DATASET_DIR = BASE_DIR / "dataset_foca_v4"
MODELS_DIR = BASE_DIR / "models"

DATASET_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("Projeto:", BASE_DIR)
print("Dataset:", DATASET_DIR)
print("Modelos:", MODELS_DIR)

Projeto: c:\Users\cooki\Documents\GitHub\FOCA\Source\AI\src\notebooks
Dataset: c:\Users\cooki\Documents\GitHub\FOCA\Source\AI\src\notebooks\dataset_foca_v4
Modelos: c:\Users\cooki\Documents\GitHub\FOCA\Source\AI\src\notebooks\models


In [5]:
caminho_dataset = Path(
    kagglehub.dataset_download(
        "lylmsc/wider-face-for-yolo-training"
    )
)

print("Dataset original:", caminho_dataset)

Dataset original: C:\Users\cooki\.cache\kagglehub\datasets\lylmsc\wider-face-for-yolo-training\versions\1


In [5]:
print(list(caminho_dataset.iterdir()))

[WindowsPath('C:/Users/cooki/.cache/kagglehub/datasets/lylmsc/wider-face-for-yolo-training/versions/1/images'), WindowsPath('C:/Users/cooki/.cache/kagglehub/datasets/lylmsc/wider-face-for-yolo-training/versions/1/labels')]


In [5]:
SEED = 42
random.seed(SEED)

In [7]:
pasta_imgs_origem = caminho_dataset / "images"
pasta_lbls_origem = caminho_dataset / "labels"

pastas = [
    "train/images",
    "train/labels",
    "val/images",
    "val/labels"
]

for pasta in pastas:
    (DATASET_DIR / pasta).mkdir(parents=True, exist_ok=True)

imagens = [
    img for img in os.listdir(pasta_imgs_origem)
    if img.lower().endswith((".jpg", ".jpeg", ".png"))
]

random.shuffle(imagens)

corte = int(len(imagens) * 0.8)

treino_imgs = imagens[:corte]
val_imgs = imagens[corte:]

print("Total:", len(imagens))
print("Treino:", len(treino_imgs))
print("Validação:", len(val_imgs))

Total: 12880
Treino: 10304
Validação: 2576


In [8]:
def copiar_arquivos(lista_arquivos, destino):
    for img_nome in lista_arquivos:
        origem_img = pasta_imgs_origem / img_nome
        destino_img = DATASET_DIR / destino / "images" / img_nome

        shutil.copy2(origem_img, destino_img)

        lbl_nome = Path(img_nome).with_suffix(".txt").name
        origem_lbl = pasta_lbls_origem / lbl_nome

        if origem_lbl.exists():
            shutil.copy2(
                origem_lbl,
                DATASET_DIR / destino / "labels" / lbl_nome
            )

copiar_arquivos(treino_imgs, "train")
copiar_arquivos(val_imgs, "val")

In [9]:
dados_yaml = {
    "path": str(DATASET_DIR.resolve()),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": ["face"]
}

caminho_yaml = DATASET_DIR / "data.yaml"

with open(caminho_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(dados_yaml, f, sort_keys=False)

print(caminho_yaml)

c:\Users\cooki\Documents\GitHub\FOCA\Source\AI\src\notebooks\dataset_foca_v4\data.yaml


In [16]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data=str(caminho_yaml),

    # Treinamento
    epochs=50,
    imgsz=800,
    batch=2,
    device=0,
    workers=0,
    amp=True,

    # Reprodutibilidade
    seed=42,
    deterministic=True,

    # Early stopping
    patience=12,

    # ----------------------------
    # DATA AUGMENTATION
    # ----------------------------

    # Iluminação / cor
    hsv_h=0.015,
    hsv_s=0.45,
    hsv_v=0.35,

    # Mudanças geométricas moderadas
    degrees=5.0,
    translate=0.10,
    scale=0.40,
    shear=2.0,
    perspective=0.0005,

    # Espelhamento horizontal
    fliplr=0.5,
    flipud=0.0,

    # Combinação de imagens
    mosaic=1.0,
    mixup=0.05,

    # Desativa mosaic nas últimas épocas
    close_mosaic=10,

    # Otimizador
    optimizer="auto",

    # Saída
    project=str(MODELS_DIR),
    name="yolov8s_foca_v4_800_50e",

    plots=True,
    save=True
)

Ultralytics 8.4.148  Python-3.11.9 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce MX570 A, 2048MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\cooki\Documents\GitHub\FOCA\Source\AI\src\notebooks\dataset_foca_v4\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.45, hsv_v=0.35, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0

KeyboardInterrupt: 

In [4]:
from ultralytics import YOLO

checkpoint = (
    MODELS_DIR
    / "yolov8s_foca_v4_800_50e"
    / "weights"
    / "last.pt"
)

model = YOLO(str(checkpoint))

model.train(
    resume=True,
    batch=1
)

New https://pypi.org/project/ultralytics/8.4.150 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.148  Python-3.11.9 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce MX570 A, 2048MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\cooki\Documents\GitHub\FOCA\Source\AI\src\notebooks\dataset_foca_v4\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.45, hsv_v=0.35, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000015F8DD7C3D0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480